In [49]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

### 1. 데이터 로드

In [14]:
# 데이터 파일 로드
df = pd.read_csv('../data/raw/data.csv', sep=';', encoding='utf-8-sig')
df.columns = [c.strip() for c in df.columns]  # 탭 섞인 컬럼명 정리

# 기본 구조 확인
print(df.shape)                # (4424, 37)
print(df.isnull().sum().sum()) # 결측치 0개
print(df.duplicated().sum())   # 중복행 0개

(4424, 37)
0
0


### 2. EDA: 이진 타겟 기준 확인

In [24]:
# Target 이진분류로 변경
df['target_binary'] = (df['Target'] == 'Dropout').astype(int)
print(df['target_binary'].value_counts(normalize=True))
# Dropout 32.12%, Non-Dropout 67.87%

target_binary
0    0.678797
1    0.321203
Name: proportion, dtype: float64


### 3. 파생 변수 생성 - Curricular units 1st sem(enrolled)

* 파생변수 만드는 법 -> df['새컬럼이름'] = 계산식

In [31]:
# 1) 수강 신청 여부
# 수강신청 과목 수가 중요한게 아니고 1학기때 수강신청을 했냐/안했냐의 구분이 필요
# Curricular units 1st sem (enrolled) 컬럼 정리
#   0: 수강신청 함 / 1: 수강신청 안함
df['zero_enrolled_1st_sem'] = (df['Curricular units 1st sem (enrolled)'] == 0).astype(int)

print(df['zero_enrolled_1st_sem'].value_counts())
print(df['zero_enrolled_1st_sem'].value_counts(normalize=True))  #   수강신청 함(96%) / 수강신청 안함(4%)

zero_enrolled_1st_sem
0    4244
1     180
Name: count, dtype: int64
zero_enrolled_1st_sem
0    0.959313
1    0.040687
Name: proportion, dtype: float64


In [32]:
# 2) 학기별 승인율
# 등록 과목이 0명인 학생은 0/0이 되어서 에러 대신 NaN(결측치)이 생김
#   --> 방지하기 위해서 "등록과목이 0보다 클 때만 나누고, 아니면 0을 넣어라"로 진행
#       np.where(조건, 조건이 참일 때 값, 거짓일 때 값)
# 1학기
df['sem1_approval_rate'] = np.where(
    df['Curricular units 1st sem (enrolled)'] > 0,
    df['Curricular units 1st sem (approved)'] / df['Curricular units 1st sem (enrolled)'],
    0
)

# 2학기
df['sem2_approval_rate'] = np.where(
    df['Curricular units 2nd sem (enrolled)'] > 0,
    df['Curricular units 2nd sem (approved)'] / df['Curricular units 2nd sem (enrolled)'],
    0
)

In [18]:
# 3) 재정 위험 점수
# 각각 True/False를 숫자(0, 1)로 바꿔서 더함 (0~3점짜리)
df['financial_risk_score'] = (
    (df['Tuition fees up to date'] == 0).astype(int) # 등록금 완납 안함 (0)
    + (df['Debtor'] == 1).astype(int)                # 채무자임 (1)
    + (df['Scholarship holder'] == 0).astype(int)    # 장학금 없음 (0)
)

In [19]:
# 4) 성적 변화량
# 음수: 성적이 떨어짐 / 양수: 성적이 오름
df['grade_change'] = df['Curricular units 2nd sem (grade)'] - df['Curricular units 1st sem (grade)']

In [ ]:
# 상관관계 확인
# 0.0 - 0.1: 거의 관계 없음
# 0.1 - 0.3: 약한 관계
# 0.3 - 0.5: 중간 정도 관계
# 0.5 -    : 강한 관계

print(f"zero_enrolled_1st_sem: {df['zero_enrolled_1st_sem'].corr(df['target_binary'])}")
# 거의 관계 없음: 선형 상관계수라는 측정 도구로는 이 변수의 가치가 안보임.
# 유지할지 뺄지 선택 필요
#   1. Logistic Regression처럼 선형관계 모델에서 도움안됨
#   2. Random Forest, XGBoost, LightGBM 같은 트리 기반 모델에서는 "4%의 희귀케이스 인데 그 안에서 이탈률 높다"는 패턴 찾을 수 있음

print(f"sem1_approval_rate: {df['sem1_approval_rate'].corr(df['target_binary'])}")      # 승인율 높으면 이탈율 감소  :: 매우 강함
print(f"sem2_approval_rate: {df['sem2_approval_rate'].corr(df['target_binary'])}")      # 승인율 높으면 이탈율 감소  :: 매우 강함
print(f"financial_risk_score: {df['financial_risk_score'].corr(df['target_binary'])}")  # 재정위험 높으면 이탈율 증가 :: 강함 
print(f"grade_change: {df['grade_change'].corr(df['target_binary'])}")                  # 성적하락 이탈율 증가      :: 약-중간

zero_enrolled_1st_sem: 0.04700513696489017
sem1_approval_rate: -0.5912359703213408
sem2_approval_rate: -0.6594127130852169
financial_risk_score: 0.43526102718106635
grade_change: -0.22533410843938487


### 4. Train/Test Split

In [33]:
X = df.drop(columns=['Target', 'target_binary'])
y = df['target_binary']

In [34]:
# 1차: 전체를 Train 60% / 나머지 40%
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=42
)

# 2차: 나머지 40%를 Val 20% / Test 20%로 절반씩 나눔
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

In [36]:
# 잘 나뉘었는지 확인
print(f"train: {X_train.shape}, val: {X_val.shape}, test: {X_test.shape}")
print(f"dropout 비율 - train: {y_train.mean():.3f}, val: {y_val.mean():.3f}, test: {y_test.mean():.3f}")

train: (2654, 41), val: (885, 41), test: (885, 41)
dropout 비율 - train: 0.321, val: 0.322, test: 0.321


### 5. 전처리 파이프라인 (ColumnTransformer)

In [ ]:
# 1) 컬럼 성격별로 분류

# 이미 0/1이라 그대로 사용
binary_cols = ['Displaced', 'Educational special needs', 'Debtor', 'Tuition fees up to date',
               'Gender', 'Scholarship holder', 'International', 'Daytime/evening attendance']

# 카테고리 종류 적음: 원-핫 인코딩
low_card_categorical = ['Marital status', 'Application mode', 'Course', 'Previous qualification']

# 카테고리 종류 많음(30~46개): 희소값 그룹화 후 원-핫
high_card_categorical = ["Mother's qualification", "Father's qualification",
                          "Mother's occupation", "Father's occupation", 'Nacionality']

# 연속형: 표준화
continuous_cols = ['Previous qualification (grade)', 'Admission grade', 'Age at enrollment',
                    'Curricular units 1st sem (grade)', 'Curricular units 2nd sem (grade)',
                    'Unemployment rate', 'Inflation rate', 'GDP',
                    'sem1_approval_rate', 'sem2_approval_rate', 'grade_change']

# 카운트형: 표준화
count_cols = ['Curricular units 1st sem (credited)', 'Curricular units 1st sem (enrolled)',
              'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (approved)',
              'Curricular units 1st sem (without evaluations)',
              'Curricular units 2nd sem (credited)', 'Curricular units 2nd sem (enrolled)',
              'Curricular units 2nd sem (evaluations)', 'Curricular units 2nd sem (approved)',
              'Curricular units 2nd sem (without evaluations)']

# 이미 만든 플래그류: 그대로 사용
flag_cols = ['zero_enrolled_1st_sem', 'financial_risk_score']

In [ ]:
# 2) 고카디널리티 컬럼 희소값 그룹화
THRESHOLD = 0.01

for col in high_card_categorical:
    # X_train에서만 비율 계산
    freq = X_train[col].value_counts(normalize=True)
    keep_categories = freq[freq >= THRESHOLD].index  # 1% 이상인 것만 "유지 목록"

    # train/val/test 모두에 같은 규칙 적용
    # -> where(조건, 대체값): 조건 참이면 원래 값 유지, 거짓이면 'Other'로 바꿈
    X_train[col] = X_train[col].astype(str).where(X_train[col].isin(keep_categories), 'Other')
    X_val[col]   = X_val[col].astype(str).where(X_val[col].isin(keep_categories), 'Other')
    X_test[col]  = X_test[col].astype(str).where(X_test[col].isin(keep_categories), 'Other')

In [40]:
# 3) ColumnTransformer 구성
# ColumnTransformer: ('이름', 처리방법, 적용할컬럼리스트) 형태로 각 그룹마다 다른 처리를 한 번에 묶어주는 것
# handle_unknown='ignore': Val/Test에 Train에서 못 본 카테고리가 있어도 에러 안 나고 그냥 0으로 처리 (안전장치)
# remainder='passthrough': 위에서 지정 안 한 나머지 컬럼(binary_cols, flag_cols)은 변형 없이 그대로 통과

numeric_all = continuous_cols + count_cols
categorical_all = low_card_categorical + high_card_categorical

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_all),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_all),
], remainder='passthrough')  # binary_cols, flag_cols는 그대로 통과

In [46]:
# 4) Train에만 fit, 전체는 transform

preprocessor.fit(X_train)  # 규칙은 Train으로만 학습

X_train_proc = preprocessor.transform(X_train)
X_val_proc = preprocessor.transform(X_val)
X_test_proc = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

In [48]:
# 5) 결과 확인
# 원본 컬럼 수(binary 8 + low_card 4 + high_card 5 + continuous 11 + count 10 + flag 2 = 40개)가 원-핫 인코딩을 거치면서 100개 넘게 늘어남
#   -> binary_cols(8) + flag_cols(2) = 10개는 그대로 통과
#   -> continuous_cols(11) + count_cols(10) = 21개는 그대로 스케일링만 되고 개수 유지
#   -> 나머지 132 - 10 - 21 = 101개가 범주형 원-핫 인코딩으로 생긴 컬럼들

print(f"최종 피처 개수: {len(feature_names)}")          # 132
print(f"X_train_proc shape: {X_train_proc.shape}")   # (2654, 132)

최종 피처 개수: 132
X_train_proc shape: (2654, 132)


### 6. 결과 저장

* train.csv, val.csv, test.csv -> 모델링 담당 팀원들이 사용할 수 있도록
* preprocesser.joblib -> 나중에 Streamlit에서 새 데이터 들어올 때 지금과 똑같이 변환하기 위해

In [50]:
# 배열(array) 상태인 걸 다시 표로 만들고, target도 같이 붙이기
train_df = pd.DataFrame(X_train_proc, columns=feature_names)
train_df['target'] = y_train.values
# y_train.values를 쓰는 이유: 
# y_train은 pandas Series라 인덱스 번호가 원본 데이터의 순서를 따라가고 있어서(예: 3, 17, 402...) 그냥 붙이면 인덱스가 안 맞을 수 있음
# .values로 순수 배열로 바꿔서 붙이면 X_train_proc(이미 인덱스 없는 배열)이랑 순서대로 매칭됨

val_df = pd.DataFrame(X_val_proc, columns=feature_names)
val_df['target'] = y_val.values

test_df = pd.DataFrame(X_test_proc, columns=feature_names)
test_df['target'] = y_test.values

# 저장
train_df.to_csv('../data/processed/train.csv', index=False)
val_df.to_csv('../data/processed/val.csv', index=False)
test_df.to_csv('../data/processed/test.csv', index=False)

joblib.dump(preprocessor, '../models/preprocessor.joblib')

print("저장 완료")

저장 완료
